<a href="https://colab.research.google.com/github/JohnDakin/COMP-475-ASSIGNMENT-CAT/blob/main/Practical_Task_2_RNN_LSTM_for_Stock_Trend_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
data = yf.download('AAPL', start='2015-01-01', end='2024-01-01')

close_prices = data['Close']

/tmp/ipykernel_450/2773071696.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('AAPL', start='2015-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


In [6]:
sequence_length = 15

X = []
y = []

for i in range(len(close_prices) - sequence_length - 1):
    seq = close_prices[i:i+sequence_length].values
    label = 1 if close_prices.iloc[i+sequence_length+1, 0] > close_prices.iloc[i+sequence_length, 0] else 0

    X.append(seq)
    y.append(label)

X = np.array(X)
y = np.array(y)

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

In [9]:
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(sequence_length, 1)),
    Dropout(0.2),

    LSTM(50),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=12,
    validation_data=(X_val, y_val)
)

Epoch 1/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.4952 - loss: 0.7035 - val_accuracy: 0.5430 - val_loss: 0.6908
Epoch 2/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4997 - loss: 0.6975 - val_accuracy: 0.4570 - val_loss: 0.7026
Epoch 3/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5175 - loss: 0.6940 - val_accuracy: 0.5430 - val_loss: 0.6909
Epoch 4/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5175 - loss: 0.6949 - val_accuracy: 0.5430 - val_loss: 0.6924
Epoch 5/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5327 - loss: 0.6928 - val_accuracy: 0.4955 - val_loss: 0.6931
Epoch 6/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4908 - loss: 0.6973 - val_accuracy: 0.5430 - val_loss: 0.6903
Epoch 7/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5054 - loss: 0.6959 - val_accuracy: 0.5430 - val_loss: 0.6903
Epoch 8/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5188 - loss: 0.6933 - val_accuracy: 0.5430 - v

In [11]:
predictions = model.predict(X_test)
predicted_labels = (predictions > 0.5).astype(int)

accuracy = accuracy_score(y_test, predicted_labels)
print("Test Accuracy:", accuracy)

11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step
Test Accuracy: 0.5443786982248521


In [12]:
baseline_predictions = np.ones_like(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.5443786982248521
